# Fixed versus semantic transcript segmentation

This notebook compares the existing fixed 2–4 minute passages with local semantic boundaries. BGE-M3 scores topic changes from cached cue-aligned atomic blocks; Qwen and BGE then receive exactly the same passages within each segmentation mode.

All segmentation runs locally. No transcript text is sent to Gemini or another cloud API. Start with a few episodes and inspect the boundary plot before scaling to the full archive.

In [ ]:
import os
from dataclasses import replace
from pathlib import Path

import pandas as pd
from IPython.display import display

from arvamusfestivali_transcripts.topic_analysis import (
    BgeM3Embedder,
    ChunkingConfig,
    DeterministicHashEmbedder,
    QwenEmbedder,
    SemanticSegmentationConfig,
    TopicModelConfig,
    build_atomic_blocks_many,
    chunk_episodes,
    embed_passages,
    evaluate_run,
    fit_topic_model,
    load_corpus,
    plot_semantic_boundaries,
    plot_topic_correspondence,
    plot_topic_sizes,
    segment_episodes_semantically,
    select_diverse_episodes,
)
from arvamusfestivali_transcripts.topic_analysis.cache import EmbeddingCache

pd.set_option("display.max_colwidth", 160)

## Configuration

In [ ]:
PROJECT_ROOT = Path(os.environ.get("TOPIC_ANALYSIS_PROJECT_ROOT", Path.cwd())).resolve()
if (
    not (PROJECT_ROOT / "data" / "transcripts").exists()
    and (PROJECT_ROOT.parent / "data" / "transcripts").exists()
):
    PROJECT_ROOT = PROJECT_ROOT.parent

YEAR = 2026
EPISODE_COUNT = 6
EXPLICIT_EPISODE_IDS: tuple[str, ...] = ()
SEGMENTATION_MODES = ("fixed", "semantic")
MODEL_KEYS = ("qwen", "bge")
DEVICE = "cpu"
BATCH_SIZES = {"qwen": 4, "bge": 4}
CACHE_ROOT = PROJECT_ROOT / "data" / "topic-analysis" / "cache"
FIXED_CHUNKING = ChunkingConfig()
SEMANTIC_CONFIG = SemanticSegmentationConfig()
SEMANTIC_CACHE_CHUNKING = ChunkingConfig(
    min_seconds=90, target_seconds=300, max_seconds=600,
    min_words=1, max_words=1_000_000, overlap_seconds=0,
    merge_tail_seconds=0, max_merged_seconds=600,
)
TOPIC_MODEL = TopicModelConfig(random_state=42)
DRY_RUN_WITH_FAKE_EMBEDDINGS = False

## Load canonical recordings

In [ ]:
episodes = load_corpus(PROJECT_ROOT / "data" / "transcripts", YEAR)
if not episodes:
    raise ValueError(f"No validated transcripts found for {YEAR}")
selected = select_diverse_episodes(
    episodes, count=EPISODE_COUNT, explicit_ids=EXPLICIT_EPISODE_IDS or None
)
display(pd.DataFrame([
    {
        "episode_id": episode.episode_id,
        "title": episode.title,
        "duration_minutes": episode.duration_seconds / 60,
        "duplicate_episode_ids": ", ".join(episode.duplicate_episode_ids),
    }
    for episode in selected
]))

## Configure local embedding adapters

In [ ]:
if DRY_RUN_WITH_FAKE_EMBEDDINGS:
    boundary_adapter = DeterministicHashEmbedder(dimension=32)
    boundary_adapter.key = "bge-boundary"
    boundary_adapter.model_id = "deterministic-hash-bge-boundary"
    adapters = {}
    for model_key in MODEL_KEYS:
        adapter = DeterministicHashEmbedder(dimension=32)
        adapter.key = model_key
        adapter.model_id = f"deterministic-hash-{model_key}"
        adapters[model_key] = adapter
    active_topic_model = TopicModelConfig(
        n_neighbors=3, n_components=2, min_cluster_size=2,
        min_samples=1, random_state=42,
    )
else:
    adapters = {
        "qwen": QwenEmbedder(device=DEVICE),
        "bge": BgeM3Embedder(device=DEVICE),
    }
    boundary_adapter = adapters["bge"]
    active_topic_model = TOPIC_MODEL

cache = EmbeddingCache(CACHE_ROOT)
print(f"Device: {DEVICE}; cache: {CACHE_ROOT}")

## Build fixed passages and semantic boundaries

In [ ]:
fixed_passages = chunk_episodes(selected, FIXED_CHUNKING)
atomic_blocks = build_atomic_blocks_many(selected, SEMANTIC_CONFIG)
boundary_embeddings = embed_passages(
    boundary_adapter,
    atomic_blocks,
    cache,
    BATCH_SIZES["bge"],
    chunking=SEMANTIC_CONFIG.atomic_chunking,
)
semantic_result = segment_episodes_semantically(
    selected,
    atomic_blocks,
    boundary_embeddings,
    SEMANTIC_CONFIG,
)
passage_sets = {
    "fixed": fixed_passages,
    "semantic": semantic_result.passages,
}

summary_rows = []
for mode, passages in passage_sets.items():
    durations = [(item.end_seconds - item.start_seconds) / 60 for item in passages]
    summary_rows.append({
        "mode": mode,
        "passages": len(passages),
        "median_minutes": pd.Series(durations).median(),
        "min_minutes": min(durations),
        "max_minutes": max(durations),
    })
display(pd.DataFrame(summary_rows))
print(
    f"Atomic BGE embeddings: {boundary_embeddings.embeddings.shape}; "
    f"{boundary_embeddings.cache_hits} cache hits"
)

## Inspect boundary scores before topic modelling

In [ ]:
first_episode = selected[0]
first_blocks = tuple(
    block for block in atomic_blocks if block.episode_id == first_episode.episode_id
)
first_boundaries = tuple(
    boundary
    for boundary in semantic_result.boundaries
    if boundary.episode_id == first_episode.episode_id
)
boundary_figures = [
    plot_semantic_boundaries(first_episode, first_blocks, first_boundaries)
]
display(boundary_figures[0])

## Embed identical passage sets with Qwen and BGE

In [ ]:
embedding_runs = {}
for mode in SEGMENTATION_MODES:
    passages = passage_sets[mode]
    cache_chunking = FIXED_CHUNKING if mode == "fixed" else SEMANTIC_CACHE_CHUNKING
    for model_key in MODEL_KEYS:
        result = embed_passages(
            adapters[model_key],
            passages,
            cache,
            BATCH_SIZES[model_key],
            chunking=cache_chunking,
        )
        embedding_runs[(mode, model_key)] = replace(
            result, model_key=f"{mode}:{model_key}"
        )
        print(
            f"{mode}:{model_key}: {result.embeddings.shape}, "
            f"{result.cache_hits} cache hits"
        )

## Fit BERTopic for every segmentation/model pair

In [ ]:
runs = {}
for mode in SEGMENTATION_MODES:
    for model_key in MODEL_KEYS:
        runs[(mode, model_key)] = fit_topic_model(
            passage_sets[mode],
            embedding_runs[(mode, model_key)],
            active_topic_model,
        )

metrics = []
for (mode, model_key), run in runs.items():
    metrics.append({
        "segmentation": mode,
        "embedding_model": model_key,
        **evaluate_run(run, embedding_runs[(mode, model_key)]),
    })
display(pd.DataFrame(metrics))

## Compare topic structure

In [ ]:
topic_size_figures = {
    key: plot_topic_sizes(run, passage_sets[key[0]])
    for key, run in runs.items()
}
correspondence_figures = {
    mode: plot_topic_correspondence(
        {
            f"{mode}:qwen": runs[(mode, "qwen")],
            f"{mode}:bge": runs[(mode, "bge")],
        },
        passage_sets[mode],
    )
    for mode in SEGMENTATION_MODES
}
for mode in SEGMENTATION_MODES:
    display(topic_size_figures[(mode, "qwen")])
    display(topic_size_figures[(mode, "bge")])
    display(correspondence_figures[mode])

## Scale up

After visually checking several boundaries, set `EPISODE_COUNT = 219`. On a CUDA-enabled machine or RunPod set `DEVICE = "cuda"` and increase local-model batch sizes gradually. Atomic BGE embeddings, fixed passages, and unchanged semantic passages are all cached independently.

In [ ]:
print(
    "SEGMENTATION_NOTEBOOK_OK "
    f"modes={','.join(SEGMENTATION_MODES)} "
    f"runs={len(runs)} boundary_figures={len(boundary_figures)}"
)